[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/43.%20The%20Security%20Inspection%20Optimization%20Problem/P43-Tier-8_executed.ipynb)

---


# P43-Tier-8: A Value-Aligned Framework for Security Decisions

## Purpose and scope

This notebook develops a small, inspectable framework for making security-inspection decisions and evaluating them against five illustrative constitutional principles. It uses a threshold-based decision rule, principle-specific validation scores, a weighted compliance score, and a simulated human-review stage. The goal is to make the decision process and its assumptions visible—not to establish that a real security system is fair, lawful, safe, or operationally effective.

The inputs are synthetic items with a risk score and descriptive fields. The outputs include an automated decision, principle scores, an aggregate compliance score, a human-review flag, summary statistics, visualizations, and an audit report. The notebook does not train a model or solve an optimization problem: its method is the explicit rule-based framework implemented below.

## Principles and modeling assumptions

The framework represents five principles: proportionality, non-discrimination, privacy preservation, transparency, and human dignity. Each principle has a weight, a constraint type, a validation function, and a threshold. Scores lie in $[0,1]$, where a larger value means stronger compliance according to the particular illustrative validation function.

The test data are generated synthetically. The risk score drives the automated action; other item attributes are included to make the records resemble inspection data, but they do not affect the decision rule. The principle checks are pedagogical proxies, not empirical measurements. In particular, returning a constant non-discrimination score cannot demonstrate that a system is non-discriminatory, and short text-based transparency scores cannot establish that a decision is genuinely explainable. No personal characteristics are used by the decision rule.

Human review is represented by a simple, seeded simulation. Its agreement rate is a property of that simulation, not a measurement of real reviewer performance or AI reliability. Processing time is an implementation-dependent measurement, not evidence of operational capacity.

## Decision rule and compliance calculation

For an item with risk score $r_i$, the automated action $a_i$ is:

$$
a_i =
\begin{cases}
\text{inspect}, & r_i > 0.8,\\
\text{flag}, & 0.5 < r_i \leq 0.8,\\
\text{clear}, & r_i \leq 0.5.
\end{cases}
$$

Let $s_{ip}$ be the score assigned to item $i$ by principle $p$, $w_p$ its weight, and $\tau_p$ its threshold. The aggregate constitutional compliance score is the weighted mean

$$
C_i = \frac{\sum_{p \in P} w_p s_{ip}}{\sum_{p \in P} w_p},
$$

where $P$ is the set of principles. A decision is sent for human review if any principle classified as `hard` has a score below its threshold:

$$
H_i = \mathbb{1}\left[\exists p \in P_{\mathrm{hard}} : s_{ip} < \tau_p\right].
$$

Here, $H_i=1$ means review is required. The weighted mean is a summary, not a substitute for the individual principle scores or hard-constraint checks. In the code, `evaluate_decision` computes $C_i$ and $H_i`; the validation functions compute $s_{ip}$.

## Workflow and interpretation

The notebook defines the data structures and principles, generates a reproducible synthetic sample, applies the decision rule, analyzes principle scores, and presents audit and human-review summaries. For $N$ items and $|P|$ principles, evaluating the decisions takes $O(N|P|)$ time because each item is checked against each principle. The resulting rates and plots describe this specific synthetic run and these chosen proxy functions; they should not be interpreted as evidence of real-world ethical compliance.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
import random
import time
from typing import Callable, Dict, List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import seaborn as sns

# Fixed seeds make the synthetic inputs and simulated human responses reproducible.
NUMPY_SEED = 43
PYTHON_SEED = 43
np.random.seed(NUMPY_SEED)
random.seed(PYTHON_SEED)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Packages imported and random seeds initialized.")
print(f"NumPy seed: {NUMPY_SEED}; Python random seed: {PYTHON_SEED}")

In [ ]:
@dataclass
class ConstitutionalPrinciple:
    """A principle and the rule used to assign its illustrative score."""

    name: str
    description: str
    weight: float
    constraint_type: str
    validation_function: Callable[["SecurityDecision"], float]
    threshold: float = 0.8


@dataclass
class SecurityDecision:
    """An item-level decision and its principle evaluation."""

    item_id: str
    decision: str
    confidence: float
    risk_score: float
    reasoning: List[str]
    ethical_scores: Dict[str, float] = field(default_factory=dict)
    human_review_required: bool = False
    timestamp: datetime = field(default_factory=datetime.now)
    constitutional_compliance: float = 0.0


@dataclass
class ConstitutionalSecurityAI:
    """Rule-based security decision framework with principle evaluation."""

    principles: List[ConstitutionalPrinciple] = field(default_factory=list)
    decisions_made: List[SecurityDecision] = field(default_factory=list)
    audit_log: List[Dict] = field(default_factory=list)
    learning_feedback: List[Dict] = field(default_factory=list)

    def add_principle(self, principle: ConstitutionalPrinciple) -> None:
        """Register a principle used to evaluate subsequent decisions."""
        if principle.weight < 0:
            raise ValueError("Principle weights must be non-negative.")
        if not 0 <= principle.threshold <= 1:
            raise ValueError("Principle thresholds must lie in [0, 1].")
        if principle.constraint_type not in {"hard", "soft", "guideline"}:
            raise ValueError("Unknown constraint type.")
        self.principles.append(principle)

    def evaluate_decision(self, decision: SecurityDecision) -> SecurityDecision:
        """Score a decision and flag any hard-principle threshold failure."""
        if not self.principles:
            raise ValueError("At least one principle is required for evaluation.")

        total_score = 0.0
        total_weight = 0.0
        hard_constraint_failed = False
        decision.ethical_scores.clear()

        for principle in self.principles:
            score = float(principle.validation_function(decision))
            if not np.isfinite(score) or not 0 <= score <= 1:
                raise ValueError(
                    f"Score for {principle.name!r} must be finite and lie in [0, 1]."
                )

            decision.ethical_scores[principle.name] = score
            if principle.constraint_type == "hard" and score < principle.threshold:
                hard_constraint_failed = True

            total_score += score * principle.weight
            total_weight += principle.weight

        if total_weight <= 0:
            raise ValueError("The sum of principle weights must be positive.")

        decision.constitutional_compliance = total_score / total_weight
        decision.human_review_required = hard_constraint_failed
        return decision

    def make_decision(self, item_data: Dict) -> SecurityDecision:
        """Apply the risk thresholds, evaluate principles, and record the result."""
        if "id" not in item_data:
            raise KeyError("Each item must contain an 'id' field.")

        risk_score = float(item_data.get("risk_score", 0.5))
        if not np.isfinite(risk_score) or not 0 <= risk_score <= 1:
            raise ValueError("Risk scores must be finite and lie in [0, 1].")

        item_type = item_data.get("type", "unknown")
        if risk_score > 0.8:
            action = "inspect"
            confidence = 0.9
        elif risk_score > 0.5:
            action = "flag"
            confidence = 0.7
        else:
            action = "clear"
            confidence = 0.8

        decision = SecurityDecision(
            item_id=str(item_data["id"]),
            decision=action,
            confidence=confidence,
            risk_score=risk_score,
            reasoning=[
                f"Risk score: {risk_score:.2f}",
                f"Item type: {item_type}",
            ],
        )
        self.evaluate_decision(decision)

        self.audit_log.append(
            {
                "timestamp": decision.timestamp,
                "item_id": decision.item_id,
                "decision": decision.decision,
                "compliance": decision.constitutional_compliance,
                "human_review": decision.human_review_required,
            }
        )
        self.decisions_made.append(decision)
        return decision

print("Decision data structures and evaluation framework are ready.")

In [ ]:
def create_constitutional_principles() -> List[ConstitutionalPrinciple]:
    """Create the five illustrative principles used in this demonstration."""

    def proportionality_check(decision: SecurityDecision) -> float:
        """Reward the action that matches the stated risk thresholds."""
        if decision.risk_score > 0.8 and decision.decision == "inspect":
            return 1.0
        if 0.5 < decision.risk_score <= 0.8 and decision.decision == "flag":
            return 1.0
        if decision.risk_score <= 0.5 and decision.decision == "clear":
            return 1.0
        return 0.7

    def non_discrimination_check(decision: SecurityDecision) -> float:
        """Return a fixed placeholder score; this does not test group fairness."""
        return 0.95

    def privacy_preservation_check(decision: SecurityDecision) -> float:
        """Assign an illustrative score based on the action's data intensity."""
        if decision.decision == "inspect":
            return 0.7
        if decision.decision == "clear":
            return 1.0
        return 0.8

    def transparency_check(decision: SecurityDecision) -> float:
        """Use reasoning length and confidence as a simple transparency proxy."""
        reasoning_score = min(len(decision.reasoning) * 0.2, 1.0)
        return (reasoning_score + decision.confidence) / 2

    def human_dignity_check(decision: SecurityDecision) -> float:
        """Assign an illustrative score based on the action taken."""
        if decision.decision == "clear":
            return 1.0
        if decision.decision == "inspect":
            return 0.8
        return 0.75

    return [
        ConstitutionalPrinciple(
            name="Proportionality",
            description="Security measures should match the stated threat level.",
            weight=0.25,
            constraint_type="hard",
            validation_function=proportionality_check,
            threshold=0.8,
        ),
        ConstitutionalPrinciple(
            name="Non-Discrimination",
            description="Decisions should not depend on protected characteristics.",
            weight=0.20,
            constraint_type="hard",
            validation_function=non_discrimination_check,
            threshold=0.9,
        ),
        ConstitutionalPrinciple(
            name="Privacy Preservation",
            description="Data collection and retention should be limited.",
            weight=0.20,
            constraint_type="soft",
            validation_function=privacy_preservation_check,
            threshold=0.7,
        ),
        ConstitutionalPrinciple(
            name="Transparency",
            description="Decisions and processes should be explainable.",
            weight=0.20,
            constraint_type="soft",
            validation_function=transparency_check,
            threshold=0.8,
        ),
        ConstitutionalPrinciple(
            name="Human Dignity",
            description="Decisions should respect individual rights and autonomy.",
            weight=0.15,
            constraint_type="hard",
            validation_function=human_dignity_check,
            threshold=0.8,
        ),
    ]


principles = create_constitutional_principles()
assert np.isclose(sum(principle.weight for principle in principles), 1.0)

print("Illustrative constitutional principles:")
for number, principle in enumerate(principles, start=1):
    print(f"{number}. {principle.name}: {principle.description}")
    print(
        f"   Weight: {principle.weight:.2f}; "
        f"type: {principle.constraint_type}; threshold: {principle.threshold:.2f}"
    )

In [ ]:
def generate_test_items(num_items: int = 50) -> List[Dict]:
    """Generate reproducible synthetic records for the demonstration."""
    if num_items < 0:
        raise ValueError("num_items must be non-negative.")

    item_types = ["container", "vehicle", "cargo", "person", "luggage"]
    origins = ["domestic", "international"]
    declared_contents = ["electronics", "clothing", "food", "machinery", "documents"]
    items = []

    for index in range(num_items):
        items.append(
            {
                "id": f"ITEM-{index + 1:03d}",
                "type": random.choice(item_types),
                "risk_score": float(np.random.beta(2, 5)),
                "origin": random.choice(origins),
                "declared_contents": random.choice(declared_contents),
            }
        )

    return items


constitutional_ai = ConstitutionalSecurityAI()
for principle in principles:
    constitutional_ai.add_principle(principle)

test_items = generate_test_items(50)

print(
    f"Framework initialized with {len(constitutional_ai.principles)} principles."
)
print(f"Generated {len(test_items)} synthetic items for evaluation.")
print("The risk score is the only item field used by the automated action rule.")

In [ ]:
def run_constitutional_simulation(
    ai_system: ConstitutionalSecurityAI,
    items: List[Dict],
) -> Dict:
    """Apply the decision rule and summarize its initial automated decisions."""
    start_time = time.perf_counter()
    decisions = [ai_system.make_decision(item) for item in items]
    elapsed_time = time.perf_counter() - start_time

    compliance_scores = [decision.constitutional_compliance for decision in decisions]
    human_reviews = [decision.human_review_required for decision in decisions]
    decision_counts = {}
    for decision in decisions:
        decision_counts[decision.decision] = (
            decision_counts.get(decision.decision, 0) + 1
        )

    return {
        "total_decisions": len(decisions),
        "avg_compliance": float(np.mean(compliance_scores)) if decisions else 0.0,
        "human_review_rate": float(np.mean(human_reviews)) if decisions else 0.0,
        "decision_distribution": decision_counts,
        "processing_time": elapsed_time,
        "decisions": decisions,
    }


simulation_results = run_constitutional_simulation(constitutional_ai, test_items)
assert simulation_results["total_decisions"] == len(test_items)
assert all(
    0 <= decision.constitutional_compliance <= 1
    for decision in simulation_results["decisions"]
)

print("Initial automated simulation summary:")
print(f"Total decisions: {simulation_results['total_decisions']}")
print(f"Mean weighted compliance score: {simulation_results['avg_compliance']:.3f}")
print(f"Human-review rate: {simulation_results['human_review_rate']:.2%}")
print(f"Decision counts: {simulation_results['decision_distribution']}")
print(f"Elapsed time: {simulation_results['processing_time']:.6f} seconds")

In [ ]:
def analyze_ethical_performance(
    decisions: List[SecurityDecision],
    principles: List[ConstitutionalPrinciple],
) -> Dict:
    """Summarize scores and threshold rates for each named principle."""
    principle_by_name = {principle.name: principle for principle in principles}
    principle_scores = {principle.name: [] for principle in principles}

    for decision in decisions:
        for principle in principles:
            if principle.name in decision.ethical_scores:
                principle_scores[principle.name].append(
                    decision.ethical_scores[principle.name]
                )

    analysis = {}
    for principle_name, scores in principle_scores.items():
        if scores:
            threshold = principle_by_name[principle_name].threshold
            analysis[principle_name] = {
                "mean_score": float(np.mean(scores)),
                "std_score": float(np.std(scores)),
                "min_score": float(np.min(scores)),
                "max_score": float(np.max(scores)),
                "threshold": threshold,
                "compliance_rate": float(
                    np.mean([score >= threshold for score in scores])
                ),
            }

    return analysis


ethical_analysis = analyze_ethical_performance(
    simulation_results["decisions"], principles
)

print("Principle score analysis for the initial automated decisions:")
for principle_name, statistics in ethical_analysis.items():
    print(f"\n{principle_name}:")
    print(
        f"  Mean score: {statistics['mean_score']:.3f} "
        f"(standard deviation: {statistics['std_score']:.3f})"
    )
    print(
        f"  Range: [{statistics['min_score']:.3f}, "
        f"{statistics['max_score']:.3f}]"
    )
    print(
        f"  Share at or above its threshold "
        f"({statistics['threshold']:.2f}): "
        f"{statistics['compliance_rate']:.2%}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    "Illustrative Security-Decision Principle Analysis",
    fontsize=16,
    fontweight="bold",
)

# The histogram shows the distribution of the weighted summary score.
compliance_scores = [
    decision.constitutional_compliance
    for decision in simulation_results["decisions"]
]
axes[0, 0].hist(
    compliance_scores,
    bins=20,
    alpha=0.7,
    color="steelblue",
    edgecolor="black",
)
if compliance_scores:
    mean_compliance = float(np.mean(compliance_scores))
    axes[0, 0].axvline(
        mean_compliance,
        color="darkred",
        linestyle="--",
        label=f"Mean: {mean_compliance:.3f}",
    )
    axes[0, 0].legend()
axes[0, 0].set_xlabel("Weighted compliance score")
axes[0, 0].set_ylabel("Number of decisions")
axes[0, 0].set_title("Weighted Score Distribution")
axes[0, 0].grid(True, alpha=0.3)

# Counts communicate the action mix without implying that one action is optimal.
decision_distribution = simulation_results["decision_distribution"]
if decision_distribution:
    axes[0, 1].pie(
        decision_distribution.values(),
        labels=decision_distribution.keys(),
        autopct="%1.1f%%",
        startangle=90,
    )
axes[0, 1].set_title("Automated Action Distribution")

# The heatmap displays individual principle scores for a small sample of decisions.
principle_names = list(ethical_analysis)
first_decisions = simulation_results["decisions"][:10]
score_matrix = np.array(
    [
        [decision.ethical_scores[name] for decision in first_decisions]
        for name in principle_names
    ]
)
if score_matrix.size:
    image = axes[0, 2].imshow(
        score_matrix,
        cmap="RdYlGn",
        aspect="auto",
        vmin=0,
        vmax=1,
    )
    axes[0, 2].set_xticks(range(len(first_decisions)))
    axes[0, 2].set_xticklabels(
        [decision.item_id for decision in first_decisions], rotation=45
    )
    axes[0, 2].set_yticks(range(len(principle_names)))
    axes[0, 2].set_yticklabels(principle_names)
    fig.colorbar(image, ax=axes[0, 2], label="Illustrative principle score")
axes[0, 2].set_title("Principle Scores for the First Decisions")
axes[0, 2].set_xlabel("Item")

# Error bars show across-decision standard deviations, not uncertainty intervals.
principle_means = [ethical_analysis[name]["mean_score"] for name in principle_names]
principle_stds = [ethical_analysis[name]["std_score"] for name in principle_names]
axes[1, 0].bar(
    principle_names,
    principle_means,
    yerr=principle_stds,
    capsize=5,
    alpha=0.75,
    color="seagreen",
    edgecolor="black",
)
axes[1, 0].set_xlabel("Principle")
axes[1, 0].set_ylabel("Mean illustrative score")
axes[1, 0].set_title("Scores by Principle")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(0, 1.05)

# The bars separate decisions flagged for review from those not flagged.
review_flags = [
    decision.human_review_required
    for decision in simulation_results["decisions"]
]
review_counts = [
    sum(review_flags),
    len(review_flags) - sum(review_flags),
]
axes[1, 1].bar(
    ["Review required", "No review flag"],
    review_counts,
    color=["darkorange", "steelblue"],
    alpha=0.75,
    edgecolor="black",
)
axes[1, 1].set_ylabel("Number of decisions")
axes[1, 1].set_title("Hard-Principle Review Flags")
axes[1, 1].tick_params(axis="x", rotation=20)
axes[1, 1].grid(True, alpha=0.3)

# Color distinguishes whether the evaluation rule requested human review.
confidences = [decision.confidence for decision in simulation_results["decisions"]]
review_colors = [
    "darkorange" if decision.human_review_required else "steelblue"
    for decision in simulation_results["decisions"]
]
axes[1, 2].scatter(
    confidences,
    compliance_scores,
    c=review_colors,
    alpha=0.7,
    s=50,
)
axes[1, 2].set_xlabel("Rule-assigned decision confidence")
axes[1, 2].set_ylabel("Weighted compliance score")
axes[1, 2].set_title("Confidence and Weighted Score")
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend(
    handles=[
        Patch(facecolor="darkorange", label="Review required"),
        Patch(facecolor="steelblue", label="No review flag"),
    ]
)

fig.tight_layout()
plt.show()
print("Plots summarize the synthetic run and the illustrative scoring rules.")

In [ ]:
def create_audit_report(ai_system: ConstitutionalSecurityAI) -> Dict:
    """Summarize the decisions recorded in the system's audit log."""
    audit_data = ai_system.audit_log
    hourly_decisions = defaultdict(int)
    hourly_compliance = defaultdict(list)

    for entry in audit_data:
        hour = entry["timestamp"].hour
        hourly_decisions[hour] += 1
        hourly_compliance[hour].append(entry["compliance"])

    hourly_average_compliance = {
        hour: float(np.mean(scores))
        for hour, scores in hourly_compliance.items()
        if scores
    }

    principle_trends = {}
    for principle in ai_system.principles:
        scores = [
            decision.ethical_scores[principle.name]
            for decision in ai_system.decisions_made
            if principle.name in decision.ethical_scores
        ]
        if scores:
            slope = (
                float(np.polyfit(np.arange(len(scores)), scores, 1)[0])
                if len(scores) > 1
                else 0.0
            )
            principle_trends[principle.name] = {
                "trend": slope,
                "current_score": float(scores[-1]),
                "average_score": float(np.mean(scores)),
            }

    return {
        "total_decisions": len(audit_data),
        "overall_compliance": (
            float(np.mean([entry["compliance"] for entry in audit_data]))
            if audit_data
            else 0.0
        ),
        "human_review_rate": (
            float(np.mean([entry["human_review"] for entry in audit_data]))
            if audit_data
            else 0.0
        ),
        "hourly_decisions": dict(hourly_decisions),
        "hourly_compliance": hourly_average_compliance,
        "principle_trends": principle_trends,
    }


audit_report = create_audit_report(constitutional_ai)
print("Audit summary for the recorded automated decisions:")
print(f"Total decisions: {audit_report['total_decisions']}")
print(f"Mean weighted compliance score: {audit_report['overall_compliance']:.3f}")
print(f"Human-review flag rate: {audit_report['human_review_rate']:.2%}")
print("Principle score trends across decision order:")

for principle_name, trend_data in audit_report["principle_trends"].items():
    trend = trend_data["trend"]
    trend_label = "increasing" if trend > 0.01 else "decreasing" if trend < -0.01 else "approximately flat"
    print(
        f"{principle_name}: final score {trend_data['current_score']:.3f}; "
        f"linear slope {trend:.4f} per decision ({trend_label})"
    )

In [ ]:
def demonstrate_human_ai_collaboration(
    ai_system: ConstitutionalSecurityAI,
    max_reviews: int = 10,
) -> Dict:
    """Simulate reviewer agreement and record any illustrative overrides."""
    if max_reviews < 0:
        raise ValueError("max_reviews must be non-negative.")

    review_required = [
        decision
        for decision in ai_system.decisions_made
        if decision.human_review_required
    ]
    reviewed_decisions = review_required[:max_reviews]
    human_overrides = 0
    human_confirmations = 0

    for decision in reviewed_decisions:
        # This probability defines a simulation assumption, not observed behavior.
        human_agrees = random.random() < 0.6

        if human_agrees:
            human_confirmations += 1
            decision.reasoning.append("Simulated reviewer confirmed the decision.")
        else:
            human_overrides += 1
            if decision.decision == "inspect":
                decision.decision = "flag"
            elif decision.decision == "flag":
                decision.decision = "clear"
            decision.reasoning.append("Simulated reviewer modified the decision.")

            # Re-score the final action so its principle scores match the recorded action.
            ai_system.evaluate_decision(decision)

        decision.timestamp = datetime.now()

        # Keep the audit entry aligned with the final action and its re-evaluation.
        matching_entries = [
            entry
            for entry in ai_system.audit_log
            if entry["item_id"] == decision.item_id
        ]
        if matching_entries:
            audit_entry = matching_entries[-1]
            audit_entry["timestamp"] = decision.timestamp
            audit_entry["decision"] = decision.decision
            audit_entry["compliance"] = decision.constitutional_compliance
            audit_entry["human_review"] = decision.human_review_required

    reviewed_count = len(reviewed_decisions)
    agreement_rate = (
        human_confirmations / reviewed_count if reviewed_count else 0.0
    )

    return {
        "total_reviewed": reviewed_count,
        "human_confirmations": human_confirmations,
        "human_overrides": human_overrides,
        "agreement_rate": agreement_rate,
    }


collaboration_metrics = demonstrate_human_ai_collaboration(constitutional_ai)
print("Simulated human-review summary:")
print(f"Decisions reviewed: {collaboration_metrics['total_reviewed']}")
print(f"Simulated confirmations: {collaboration_metrics['human_confirmations']}")
print(f"Simulated overrides: {collaboration_metrics['human_overrides']}")
print(
    "Agreement rate in this seeded simulation: "
    f"{collaboration_metrics['agreement_rate']:.2%}"
)
print("This agreement rate is not an estimate of real reviewer performance.")

In [ ]:
print("=" * 72)
print("P43-Tier-8: Value-Aligned Security-Decision Demonstration")
print("=" * 72)

print("\nRun summary:")
print(f"Principles represented: {len(constitutional_ai.principles)}")
print(f"Synthetic items processed: {simulation_results['total_decisions']}")
print(
    "Initial mean weighted compliance score: "
    f"{simulation_results['avg_compliance']:.3f}"
)
print(
    "Initial human-review flag rate: "
    f"{simulation_results['human_review_rate']:.2%}"
)
print(
    "Elapsed time for the initial simulation: "
    f"{simulation_results['processing_time']:.6f} seconds"
)

print("\nInitial principle analysis:")
for principle_name, statistics in ethical_analysis.items():
    print(
        f"{principle_name}: mean score {statistics['mean_score']:.3f}; "
        f"share at or above its threshold {statistics['compliance_rate']:.1%}"
    )

print("\nHuman-review simulation:")
print(
    f"Agreement among reviewed decisions: "
    f"{collaboration_metrics['agreement_rate']:.1%}"
)

print("\nInterpretation:")
print(
    "The framework makes its threshold rule, principle scores, and review flags "
    "visible for inspection."
)
print(
    "Its numerical scores depend on illustrative proxy functions and synthetic "
    "inputs; they do not certify real-world ethical or legal compliance."
)
print(
    "A constant non-discrimination score does not measure group fairness, and "
    "the simulated reviewer agreement rate is not empirical evidence."
)
print("=" * 72)